## connecting to the DataBase

In [1]:
import sqlite3 # Standard Python library for SQLite database interaction 
import requests # Library for making HTTP requests (to download the SQL script) 
from langchain_community.utilities.sql_database import SQLDatabase # LangChain utility to interact with SQL databases 
from sqlalchemy import create_engine # SQLAlchemy function to create a database engine 
from sqlalchemy.pool import StaticPool # SQLAlchemy connection pool class for in-memory databases 
 
def get_engine_for_chinook_db(): 
    """Pull sql file, populate in-memory database, and create engine.""" 
    # URL to the raw SQL script for the Chinook database 
    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql" 
     
    # Fetch the SQL script content from the URL 
    response = requests.get(url) 
    sql_script = response.text 
 
    # Create an in-memory SQLite database connection. 
    # `check_same_thread=False` is important for SQLAlchemy's StaticPool. 
    connection = sqlite3.connect(":memory:", check_same_thread=False) 
     
    # Execute the SQL script to populate the in-memory database with Chinook data 
    connection.executescript(sql_script) 
     
    # Create a SQLAlchemy engine for the in-memory SQLite database. 
    # `creator=lambda: connection` tells SQLAlchemy how to get a new connection. 
    # `poolclass=StaticPool` is used for in-memory databases, ensuring the same connection is reused. 
    # `connect_args` are passed directly to the `sqlite3.connect` function. 
    return create_engine( 
        "sqlite://", 
        creator=lambda: connection, 
        poolclass=StaticPool, 
        connect_args={"check_same_thread": False}, 
    ) 
 
# Get the SQLAlchemy engine for our Chinook database 
engine = get_engine_for_chinook_db() 
# Create a LangChain SQLDatabase utility instance from the engine. 
# This utility will help our agents interact with the database via SQL queries. 
db = SQLDatabase(engine)

C:\Users\mohammed\AppData\Local\Temp\ipykernel_4996\879410195.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities.sql_database import SQLDatabase # LangChain utility to interact with SQL databases


## making the Graph State

In [2]:
from typing_extensions import TypedDict # For defining dictionaries with type hints 
from typing import Annotated, List # For type hinting lists and adding annotations 
from langgraph.graph.message import AnyMessage, add_messages # For managing messages in the graph state 
from langgraph.managed.is_last_step import RemainingSteps # For tracking recursion limits 
 
class State(TypedDict): 
    """Represents the state of our LangGraph agent.""" 
    # customer_id: Stores the unique identifier for the current customer. 
    customer_id: str 
     
    # messages: A list of messages that form the conversation history. 
    # Annotated with `add_messages` to ensure new messages are appended rather than overwritten. 
    messages: Annotated[list[AnyMessage], add_messages] 
     
    # loaded_memory: Stores information loaded from the long-term memory store,  
    # typically user preferences or historical context. 
    loaded_memory: str

## Music Catalog Sub-Agent

In [3]:
from langchain_core.tools import tool
from sqlalchemy import text

@tool
def get_albums_by_artist(artist: str):
    """Query the database to retrieve albums by a specific artist."""
    query = text("""
        SELECT Album.Title FROM Album
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Artist.Name LIKE '%' || :artist || '%'
    """)
    with engine.connect() as conn:
        result = conn.execute(query, {"artist": artist})
        albums = [row[0] for row in result.fetchall()]
    if not albums:
        return f"No albums found for artist '{artist}'."
    return albums

@tool
def get_tracks_by_artist(artist: str):
    """Query the database to retrieve tracks by a specific artist."""
    query = text("""
        SELECT Track.Name 
        FROM Track 
        JOIN Album ON Track.AlbumId = Album.AlbumId 
        JOIN Artist ON Album.ArtistId = Artist.ArtistId 
        WHERE Artist.Name LIKE '%' || :artist || '%'
    """)
    with engine.connect() as conn:
        result = conn.execute(query, {"artist": artist})
        tracks = [row[0] for row in result.fetchall()]
    if not tracks:
        return f"No tracks found for artist '{artist}'."
    return tracks

@tool
def get_songs_by_genre(genre: str): 
    """Query the database to retrieve songs by a specific genre."""
    query = text("""
        SELECT Track.Name 
        FROM Track 
        JOIN Genre ON Track.GenreId = Genre.GenreId 
        WHERE Genre.Name = :genre
    """)
    with engine.connect() as conn:
        result = conn.execute(query, {"genre": genre})
        songs = [row[0] for row in result.fetchall()]
    return songs if songs else f"No songs found for genre '{genre}'."

@tool
def check_for_songs(song_title: str): 
    """Check if a song exists in the database by its title."""
    query = text("SELECT COUNT(*) FROM Track WHERE Name = :song_title")
    with engine.connect() as conn:
        result = conn.execute(query, {"song_title": song_title})
        count = result.fetchone()[0]
    return count > 0

In [4]:
import os
import getpass
from pathlib import Path
from dotenv import load_dotenv

env_file = next(
        (parent / ".env" for parent in (Path.cwd(), *Path.cwd().parents) if (parent / ".env").is_file()),
        None,
    )
if env_file:
    load_dotenv(env_file, override=True)
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Please provide your GROQ_API_KEY: ")

In [5]:
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq

music_catalog_agent = create_react_agent(
  model=ChatGroq(model="openai/gpt-oss-20b", temperature=0),
  tools=[get_albums_by_artist, get_tracks_by_artist, get_songs_by_genre, check_for_songs],
  prompt="""
            You are the music catalog specialist for a digital music store.
            Use your tools to answer questions about albums, tracks, genres, and song availability. 
            Only rely on tool results — never invent catalog data. 
            If an artist, song, or genre isn't found, say so plainly instead of guessing.
        """,
  name="music_catalog_agent",
)

C:\Users\mohammed\AppData\Local\Temp\ipykernel_4996\1844184477.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  music_catalog_agent = create_react_agent(


## Invoice Information Sub-Agent

In [6]:
from langchain_core.tools import tool
from sqlalchemy import text

@tool
def get_invoices_by_customer_sorted_by_date(customer_id: str) -> list[dict]:
    """Query the database to retrieve invoices for a specific customer, sorted by date."""
    query = text("""
        SELECT InvoiceId, InvoiceDate, Total 
        FROM Invoice 
        WHERE CustomerId = :customer_id 
        ORDER BY InvoiceDate ASC
    """)
    
    with engine.connect() as conn:
        result = conn.execute(query, {"customer_id": customer_id})
        invoices = [{"InvoiceId": row[0], "InvoiceDate": row[1], "Total": row[2]} for row in result.fetchall()]
    return invoices if invoices else f"No invoices found for customer ID '{customer_id}'."

@tool
def get_invoices_sorted_by_unit_price(customer_id: str) -> list[dict]:
    """Query the database to retrieve invoices for a specific customer, sorted by unit price."""
    query = text("""
        SELECT InvoiceId, InvoiceDate, Total 
        FROM Invoice 
        WHERE CustomerId = :customer_id 
        ORDER BY Total ASC
    """)
    
    with engine.connect() as conn:
        result = conn.execute(query, {"customer_id": customer_id})
        invoices = [{"InvoiceId": row[0], "InvoiceDate": row[1], "Total": row[2]} for row in result.fetchall()]
    return invoices if invoices else f"No invoices found for customer ID '{customer_id}'."

@tool
def get_employee_by_invoice_and_customer(invoice_id: str, customer_id: str) -> dict:
    """Query the database to retrieve the employee associated with a specific invoice and customer."""
    query = text("""
        SELECT Employee.EmployeeId, Employee.FirstName, Employee.LastName 
        FROM Employee 
        JOIN Customer ON Employee.EmployeeId = Customer.SupportRepId 
        JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId 
        WHERE Invoice.InvoiceId = :invoice_id AND Customer.CustomerId = :customer_id
    """)
    
    with engine.connect() as conn:
        result = conn.execute(query, {"invoice_id": invoice_id, "customer_id": customer_id})
        row = result.fetchone()
        
    if row:
        return {"EmployeeId": row[0], "FirstName": row[1], "LastName": row[2]}
    else:
        return {}

In [7]:
invoice_agent = create_react_agent(
  model=ChatGroq(model="openai/gpt-oss-20b", temperature=0),
  tools=[get_invoices_by_customer_sorted_by_date, get_invoices_sorted_by_unit_price, get_employee_by_invoice_and_customer],
  prompt="""
          You are the invoice specialist for a digital music store. 
          Use your tools to answer questions about a customer's past purchases, invoice totals, invoice dates, and which employee supports their account. 
          You will always be given a customer_id in the conversation context — use that exact value, never ask the customer for it, and never guess one. 
          Only rely on tool results.
      """,
  name="invoice_agent",
)

C:\Users\mohammed\AppData\Local\Temp\ipykernel_4996\1293773947.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  invoice_agent = create_react_agent(


## the Supervisor agent

In [8]:
from langgraph_supervisor import create_supervisor

supervisor = create_supervisor(
    model=ChatGroq(model="openai/gpt-oss-20b", temperature=0),
    agents=[music_catalog_agent, invoice_agent],
    prompt="""
          You are the supervisor for a digital music store's customer support system. 
          You manage two specialists: music_catalog_agent (albums, tracks, genres, song availability) and 
          invoice_agent (past purchases, invoice totals, support employee info). 
          Decide which specialist(s) a query needs and delegate — never answer catalog or invoice questions yourself. 
          If a query needs both, delegate to both before replying. 
          Always pass along the customer_id given in the conversation context so invoice_agent can use it. 
          Combine the specialists' answers into one clear reply.
          Never attempt to call music catalog or invoice database tools directly. 
          You MUST use the transfer tools to delegate the user's request to the appropriate specialist.
      """,
    add_handoff_back_messages=True,
    output_mode="full_history",
).compile()

In [9]:
supervisor.invoke({"messages": [{"role": "user", "content": "Customer ID: 1. What albums do you have by AC/DC?"}]})

{'messages': [HumanMessage(content='Customer ID: 1. What albums do you have by AC/DC?', additional_kwargs={}, response_metadata={}, id='ddadc8dc-1f0c-4f71-82e2-d8114bff90f1'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to delegate to music_catalog_agent.', 'tool_calls': [{'id': 'fc_024da4b7-c094-489c-81b3-990435fab27d', 'function': {'arguments': '{}', 'name': 'transfer_to_music_catalog_agent'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 312, 'total_tokens': 344, 'completion_time': 0.032304289, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.018687269, 'prompt_tokens_details': None, 'queue_time': 0.609067202, 'total_time': 0.050991558}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c9afb2bdb4', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, name='supervisor', id='lc_run--01a10225-be10-71d2-947d-3eed4b8bd5

## verify info node and human in the loop node

In [10]:
import re
from sqlalchemy import text

def extract_identifier(text: str) -> str | None:
    if not text:
        return None

    email_match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    if email_match:
        return email_match.group(0)

    phone_match = re.search(r'\+?[\d\s\-\(\)]{7,20}', text)
    if phone_match:
        return phone_match.group(0).strip()

    id_match = re.search(r'\b\d+\b', text)
    if id_match:
        return id_match.group(0)

    return None

def lookup_customer_id(identifier: str) -> str | None:
    """Query the Chinook DB to find a CustomerId matching the identifier."""
    query = text("""
        SELECT CustomerId 
        FROM Customer 
        WHERE Email = :identifier 
           OR Phone = :identifier 
           OR CAST(CustomerId AS TEXT) = :identifier
    """)
    
    with engine.connect() as conn:
        result = conn.execute(query, {"identifier": identifier}).fetchone()
        
    if result:
        return str(result[0])
    return None

In [11]:
print(extract_identifier("My phone number is +55 (12) 3923-5555. How much was my most recent purchase?"))
# should print the phone number, not "55"

print(extract_identifier("How much was my most recent purchase?"))
# should print None

+55 (12) 3923-5555
None


In [12]:
from langgraph.types import Command, interrupt
from langchain_core.messages import HumanMessage
from typing import Literal

def verify_info(state: State) -> Command[Literal["human_input", "load_memory"]]:
    if state.get("customer_id"):
        return Command(goto="load_memory")

    last_message = state["messages"][-1]
    identifier = extract_identifier(last_message.content)
    customer_id = lookup_customer_id(identifier) if identifier else None

    if customer_id:
        return Command(update={"customer_id": customer_id}, goto="load_memory")
    return Command(goto="human_input")

def human_input(state: State) -> Command[Literal["verify_info"]]:
    user_reply = interrupt("Please provide your Customer ID, email, or phone number.")
    return Command(
        update={"messages": [HumanMessage(content=user_reply)]},
        goto="verify_info",
    )

## load_memory node and create_memory node

In [13]:
from langchain_core.messages import SystemMessage
from langgraph.store.base import BaseStore
from langgraph.types import Command
from typing import Literal

def load_memory(state: State, *, store: BaseStore) -> Command[Literal["supervisor"]]:
    customer_id = state.get("customer_id")
    namespace = ("preferences", customer_id)
    memory_item = store.get(namespace, "music_prefs")
    memory = memory_item.value if memory_item else "No specific preferences saved yet."

    context_message = SystemMessage(
        content=f"Customer ID: {customer_id}. Known preferences: {memory}"
    )
    return Command(
        update={"loaded_memory": memory, "messages": [context_message]},
        goto="supervisor",
    )

In [14]:
from pydantic import BaseModel, Field

class PreferenceExtraction(BaseModel):
    has_music_preference: bool = Field(
        description="True ONLY if the message expresses a NEW music taste (genre, artist, mood) that is not already captured in the existing preferences. False if the preference is already known, or for unrelated questions."
    )
    preference: str | None = Field(
        default=None, 
        description="A short phrase capturing the NEW preference. Must be null if the preference is already listed in the existing preferences."
    )

preference_extractor = ChatGroq(model="openai/gpt-oss-20b", temperature=0).with_structured_output(PreferenceExtraction)

def create_memory(state: State, *, store: BaseStore) -> Command[Literal["__end__"]]:
    customer_id = state.get("customer_id")
    messages = state.get("messages", [])
    last_user_message = next((m.content for m in reversed(messages) if m.type == "human"), "")

    if customer_id and last_user_message:
        # 1. Fetch existing preferences FIRST
        namespace = ("preferences", customer_id)
        current_item = store.get(namespace, "music_prefs")
        current_memory = current_item.value if current_item else "No existing preferences."

        # 2. Pass the existing memory to the LLM in the prompt
        prompt = (
            f"Existing Preferences: {current_memory}\n"
            f"User Message: \"{last_user_message}\"\n\n"
            f"Does this message express a NEW music preference not already covered by the Existing Preferences?"
        )
        
        extraction = preference_extractor.invoke(prompt)

        # 3. Only update if the LLM determines it's a new preference
        if extraction.has_music_preference and extraction.preference:
            if current_memory == "No existing preferences.":
                new_memory = extraction.preference
            else:
                new_memory = f"{current_memory} | {extraction.preference}"
                
            store.put(namespace, "music_prefs", new_memory)

    return Command(goto="__end__")

In [15]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.graph import StateGraph, START

# Initialize both the checkpointer (for human-in-the-loop) and the store (for long-term memory)
memory_saver = MemorySaver()
store = InMemoryStore()

# Initialize the master graph
workflow = StateGraph(State)

# Add all nodes
workflow.add_node("verify_info", verify_info)
workflow.add_node("human_input", human_input)
workflow.add_node("load_memory", load_memory)
workflow.add_node("supervisor", supervisor)
workflow.add_node("create_memory", create_memory)

# Define the graph edges
workflow.add_edge(START, "verify_info")
workflow.add_edge("supervisor", "create_memory")

# Compile the graph with both systems active
app = workflow.compile(checkpointer=memory_saver, store=store)

In [16]:
import uuid
from langgraph.types import Command

def new_session():
    return {"configurable": {"thread_id": str(uuid.uuid4())}}

def is_paused(graph, config):
    return bool(graph.get_state(config).next)

def extract_reply(result):
    if "__interrupt__" in result:
        return result["__interrupt__"][0].value
    ai_messages = [m for m in result["messages"] if m.type == "ai"]
    return ai_messages[-1].content if ai_messages else "(no response)"

def chat(graph, config, user_text):
    if is_paused(graph, config):
        result = graph.invoke(Command(resume=user_text), config=config)
    else:
        result = graph.invoke({"messages": [{"role": "user", "content": user_text}]}, config=config)
    return extract_reply(result)

In [17]:
config1 = new_session()
print(chat(app, config1, "My phone number is +55 (12) 3923-5555. How much was my most recent purchase? What albums do you have by the Rolling Stones?"))

Your most recent purchase (Invoice 382) was for **$8.91** on 2025‑08‑07.

Here are the Rolling Stones albums we have in our catalog:

1. **Hot Rocks, 1964‑1971 (Disc 1)**
2. **No Security**
3. **Voodoo Lounge**

If you’d like to see more details about any of these albums or explore other titles, just let me know!


In [18]:
print(chat(app, config1, "List some songs that match my preferences?"))

Here are some Rolling Stones tracks available in our catalog:

- “Satisfaction”  
- “Paint It Black”  
- “Ruby Tuesday”  
- “Under My Thumb”  
- “Time Is On My Side”  
- “Heart Of Stone”  
- “Play With Fire”  
- “Get Off Of My Cloud”  
- “Mother's Little Helper”  
- “19th Nervous Breakdown”


In [19]:
config2 = new_session()
print(chat(app, config2, "How much was my most recent purchase? What albums do you have by the Rolling Stones?"))

Please provide your Customer ID, email, or phone number.


In [20]:
print(chat(app, config2, "My phone number is +55 (12) 3923-5555."))

Your most recent purchase was **$8.91** on **August 7, 2025** (Invoice 382).

**Albums we have by the Rolling Stones**

1. *Hot Rocks, 1964‑1971 (Disc 1)*
2. *No Security*
3. *Voodoo Lounge*

Let me know if you’d like details on any of these albums or if there’s anything else I can help with!


In [ ]:
# A — Human-in-the-loop with an INCORRECT answer, then a correct one (not just missing)
config3 = new_session()
print(chat(app, config3, "What albums do you have?"))                  # interrupts: no info given
print("="* 20)
print(chat(app, config3, "my email is notreal@nowhere.com"))           # interrupts again: no match found
print("="* 20)
print(chat(app, config3, "3"))                                          # resolves — use any real CustomerId 1-59

Please provide your Customer ID, email, or phone number.
Please provide your Customer ID, email, or phone number.
I can list albums for a specific artist. Could you let me know which artist’s albums you’d like to see?


In [24]:
# B — Invoice-only query, multi-tool chaining within one agent
config4 = new_session()
print(chat(app, config4, "My customer id is 3. Sort my invoices by price, tell me the highest one, and who was the support employee for it?"))

**Customer ID 3 – Invoice Summary**

| Invoice ID | Date       | Total | Support Employee |
|------------|------------|-------|------------------|
| **110**    | 2022‑04‑21 | **$13.86** | Jane Peacock (Employee #3) |

- **Highest‑priced invoice:** Invoice 110, amounting to **$13.86**.  
- **Support employee:** Jane Peacock, Employee ID 3.


In [25]:
# C — Music-only query, exercising get_songs_by_genre + check_for_songs together
config5 = new_session()
print(chat(app, config5, "My customer id is 5. Do you have any Jazz songs? Also, is there a song called 'Highway to Hell'?"))
print("")
print("="* 30)
print("")

Here’s what we have for you:

**Jazz selections**  
We have a wide range of Jazz tracks available. Some highlights include:

- “Desafinado”  
- “Garota De Ipanema”  
- “Samba De Uma Nota Só (One Note Samba)”  
- “Por Causa De Você”  
- “Ligia”  
- “Fotografia”  
- “Dindi (Dindi)”  
- “Se Todos Fossem Iguais A Você (Instrumental)”  
- “Falando De Amor”  
- “Angela”  
- “Corcovado (Quiet Nights Of Quiet Stars)”  
- “Outra Vez”  
- “O Boto (Bôto)”  
- “Canta, Canta Mais”  
- “Quadrant”  
- “Snoopy's search‑Red baron”  
- “Spanish moss‑\"A sound portrait\"‑Spanish moss”  
- “Moon germs”  
- “Stratus”  
- “The pleasant pheasant”  
- “Solo‑Panhandler”  
- “Do what cha wanna”  
- “Heart of the Night”  
- “De La Luz”  
- “Westwood Moon”  
- “Midnight”  
- “Playtime”  
- “Surrender”  
- “Valentino's”  
- “Believe”  
- “As We Sleep”  
- “When Evening Falls”  
- “J Squared”  
- “Best Thing”  
- “Now's The Time”  
- “Jeru”  
- “Compulsion”  
- “Tempus Fugit”  
- “Walkin'”  
- “'Round Midnight”  
-

In [26]:
# D — Long-term memory ACROSS a brand-new thread (the real test of Task 6)
config6 = new_session()
print(chat(app, config6, "My phone number is +55 (12) 3923-5555. Recommend me some music I might like."))

Here are the Rolling Stones albums and tracks you might enjoy:

**Albums**
- Hot Rocks, 1964-1971 (Disc 1)
- No Security
- Voodoo Lounge

**Tracks**
- Time Is On My Side
- Heart Of Stone
- Play With Fire
- Satisfaction
- As Tears Go By
- Get Off Of My Cloud
- Mother's Little Helper
- 19th Nervous Breakdown
- Paint It Black
- Under My Thumb
- Ruby Tuesday
- Let's Spend The Night Together
- Intro
- You Got Me Rocking
- Gimmie Shelters
- Flip The Switch
- Memory Motel
- Corinna
- Saint Of Me
- Wainting On A Friend
- Sister Morphine
- Live With Me
- Respectable
- Thief In The Night
- The Last Time
- Out Of Control
- Love Is Strong
- You Got Me Rocking
- Sparks Will Fly
- The Worst
- New Faces
- Moon Is Up
- Out Of Tears
- I Go Wild
- Brand New Car
- Sweethearts Together
- Suck On The Jugular
- Blinded By Rainbows
- Baby Break It Down
- Thru And Thru
- Mean Disposition
